# Aviso de bloqueio por inadimplência — Cta Smart

Nada é enviado até a célula **6**, e mesmo lá só para você.

| # | Etapa | Envia? |
|---|---|---|
| 1 | Traz o código do GitHub | não |
| 2 | Pré-voo | não |
| 3 | Define a campanha | não |
| 4 | **Régua**: calcula quem entra | não |
| 5 | `preparar()` — congela | não |
| 6 | Teste: tudo para você | só para você |
| 7 | Produção | **sim** |
| 8 | Painel | não |
| 9 | Reconciliar (no prazo) | não |
| 10 | Cards do Pipefy | não |


## 1. Código

In [ ]:
import os, shutil, subprocess, sys

USUARIO, REPO = "yagoppereira", "Automa-o-Bloqueio"
DIR, SECRET   = "/content/campanha", "rotinabloqueio"

MODULOS = ["agrupamento","campanha_v2","encargos","entrada","envio","io_sheets",
           "main","painel","pipefy","preflight","template","vendedores"]

def atualizar_codigo():
    from google.colab import userdata
    try:
        token = userdata.get(SECRET)
    except userdata.SecretNotFoundError:
        print(f"Secret '{SECRET}' não existe (ícone da chave)."); return False
    except userdata.NotebookAccessError:
        print(f"Secret '{SECRET}' sem acesso a este notebook."); return False

    if os.path.exists(DIR):
        shutil.rmtree(DIR)
    r = subprocess.run(["git","clone","--depth","1","-q",
                        f"https://{token.strip()}@github.com/{USUARIO}/{REPO}.git", DIR],
                       capture_output=True, text=True)
    if r.returncode:
        print("clone falhou:", (r.stderr or "").replace(token,"***")[:300]); return False

    subprocess.run([sys.executable,"-m","pip","install","-q","-r",
                    f"{DIR}/requirements.txt"], check=False)
    if DIR not in sys.path:
        sys.path.insert(0, DIR)
    for m in MODULOS:            # sem isto, o Colab roda o código antigo após um push
        sys.modules.pop(m, None)
    print(f"código atualizado ({len(os.listdir(DIR))} arquivos)")
    return True

atualizar_codigo()

## 2. Pré-voo

Confere módulos, paridade do cálculo, planilha, abas, queries e escrita.
Não envia nada.

In [ ]:
import preflight
preflight.rodar()

## 3. A campanha

`data_envio` é a base do cálculo de juros. `data_limite` é o prazo no texto.

In [ ]:
from datetime import date
import main, campanha_v2

gc, bq = main.conectar()

c = campanha_v2.Campanha(
    id_campanha="2026-09-A",
    data_envio=date.today(),
    data_limite=date(2026, 9, 15),
    copia_padrao=("contato.financeiro@ctasmart.com.br",),
    teto_diario=450,
)
print(f"{c.id_campanha} | modo {c.modo} | prazo {c.dias_de_prazo} dias")

## 4. A régua

**deve a X frequência E tem uso recente → entra.**

Frequência = meses distintos com título vencido, no grupo (raiz de CNPJ),
incluindo X90. Clientes em X91 (cobrança jurídica) e X92 (recuperação
judicial) são excluídos automaticamente — não é preferência, é restrição.

Escreve a `Campanha_Input`. **Deixe `acao` em branco para o cliente entrar**;
preencha só para vetar, com o motivo. Quem tem frequência mas não tem alocação
fica de fora (uso desconhecido) e vai para `Triagem_Sem_Alocacao`.

In [ ]:
cands = main.candidatos(gc, bq, freq_minima=7, dias_uso=90)

### Confira antes de seguir

Abra a `Campanha_Input`. Vete quem estiver em negociação, acordo ou for conta
sensível. Olhe também `Triagem_Sem_Alocacao` — são clientes endividados que a
régua não alcança porque a bomba deles não está alocada.

## 5. Preparar (congela)

Lê a aba, resolve destinatários e vendedores, calcula encargos e **congela**
tudo. Escreve `Campanha_Previa`, `Campanha_Titulos`, `Campanha_Bombas` e
`Triagem`. Não existe caminho de envio aqui dentro.

In [ ]:
resultado = main.preparar(c, gc, bq)

## 6. Teste — tudo para você

Cada mensagem chega com os destinatários **reais** impressos no topo.
Senha de app em Secrets do Colab, nunca colada na célula.

In [ ]:
import envio
from google.colab import userdata

c_teste = campanha_v2.Campanha.teste(
    id_campanha=c.id_campanha, data_envio=c.data_envio, data_limite=c.data_limite,
    para="yago.gutterres@ctasmart.com.br",
    copia_padrao=c.copia_padrao, teto_diario=c.teto_diario,
)
enviar = envio.criar_enviador_smtp(
    remetente="yago.gutterres@ctasmart.com.br",
    senha_app=userdata.get("SMTP_APP_PASSWORD"),
    nome_exibicao="Cta Smart - Cobrança",
)
main.disparar(c_teste, gc, enviar)

## 7. Produção

**Só depois de ler os e-mails de teste.** `confirmo_producao=True` é
obrigatório. Se cair no meio, rode de novo: o log é gravado por mensagem e a
execução seguinte pula quem já recebeu.

In [ ]:
c_prod = campanha_v2.Campanha(
    id_campanha=c.id_campanha, data_envio=c.data_envio, data_limite=c.data_limite,
    copia_padrao=c.copia_padrao, teto_diario=c.teto_diario,
    dry_run=False, confirmo_producao=True,
)
main.disparar(c_prod, gc, enviar)

## 8. Painel

Onde cada cliente está agora. A etapa é **derivada** do log, do saldo e do
serial: campo de status preenchido à mão envelhece e passa a mentir.

In [ ]:
import painel, io_sheets as io

p1 = io.abrir(gc, main.ID_DESTINO)
linhas = painel.montar(
    previa=[l for l in io.ler_aba(p1,"Campanha_Previa") if l["id_campanha"]==c.id_campanha],
    log=io.ler_aba(p1,"Campanha_Log", obrigatoria=False),
    campanha=c,
    bombas_snapshot=[l for l in io.ler_aba(p1,"Campanha_Bombas") if l["id_campanha"]==c.id_campanha],
)
painel.escrever(p1, linhas, io)

## 9. Reconciliar (no dia do prazo)

Única releitura do BigQuery depois do congelamento. Pagamento parcial é
decisão sua.

In [ ]:
rec = main.reconciliar(c, gc, bq)

## 10. Cards do Pipefy

Texto dos cards a partir do **snapshot**, nunca de consulta nova: serial some
da view quando o bloqueio é concluído. Quem quitou não entra.

In [ ]:
codigos = [r["codigo_cliente"] for r in rec["mantem"]]
cards = main.gerar_cards(c, gc, codigos)

for card in cards:
    print("=" * 70)
    print(f"{card['titulo']}  |  {card['cnpj']}\nTipo: {card['tipo_solicitacao']}")
    print(card["observacoes"])